# Provisioning resources

## Prerequisites

Before starting, make sure you:

- Have completed the [Preparation](./00_preparation.ipynb) tutorial.
- Have a basic understanding of Python.

## Introduction

5G experiments run on physical or virtual infrastructure resources. To set up and configure these resources to host 5G workloads, the framework relies on **resource providers** and **provisioners**.

### Core Concepts

* **Metamodel:** Describes the abstract 5G network topology and network functions. Because metamodels are implementation-agnostic, you can easily share and run your 5G network definitions across different environments and protocol implementations.
* **Deployment Model:** Complements the abstract metamodel by supplying the technical parameters required for a specific physical or virtual deployment.
* **Provisioners:** Explicitly declare the infrastructure nodes that will host 5G functions. By listing and tagging machines with appropriate roles in the `nodes` section of the deployment model, the orchestrator library automatically schedules network functions onto compatible nodes.

The `ifran` orchestrator offers a collection of built-in provisioners tailored for various resource providers, each with its own specific configuration parameters.

### Tutorial Scope

This tutorial focuses exclusively on **infrastructure provisioning** and does not deploy an active 5G network. Nevertheless, because provisioners always require a valid metamodel reference, you will instantiate and use an empty metamodel throughout this guide.

In [1]:
from ifran import MetaModel

metamodel = MetaModel()

Provisioners are managed using instances of the `ProvisioningManagement` class, which automatically determine how to configure and provision resources based on the supplied deployment model.

## `null` Provisioner

The `null` provisioner handles pre-existing infrastructure. It indicates that no automated provisioning action is required for the declared nodes; instead, it assumes that the nodes are already active, running, and fully configured to host workloads orchestrated by the library.

### Manual Provisioning Flag

Because no automated provider operations take place, you must explicitly inform the library that a node is available by setting the `is_provisioned` attribute to `true`. When using automated provisioners, you do not need to manage this attribute manually, as the provisioner automatically populates it based on the provisioning status.

### SSH Access & Authentication Configurations

The library always communicates with target nodes over SSH. Below are common authentication patterns:

* **Default Configuration (`default.example.org`):** Accessed using default SSH parameters. By default, the username is `root`, and the client leverages the default keys and settings of the local user running the library.
* **Sudo User (`password.example.org`):** Connected via the `ubuntu` user using default SSH keys. Because administrative privileges require elevation, the configuration explicitly supplies the `sudo` password for the `ubuntu` user.
* **Custom SSH Key (`machine-key`):** Used when a node cannot be accessed via the local user's default SSH keys. A path to a specific private key is explicitly provided.

> **Important:** When using the `null` provisioner, if no hostname is provided, the the node name in the metamodel is sued as the machine's actual, resolvable hostname or IP address to establish SSH connections.

In [2]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
nodes:
  default.example.org:
    is_provisioned: true
    provisioner: null

  password.example.org:
    is_provisioned: true
    provisioner: null
    ssh_info:
      user: ubuntu
      sudo_password: password

  machine-key:
    is_provisioned: true
    ssh_info:
      hostname: key.example.org
      private_key_file: /home/user/.ssh/id_ed25519_custom
"""
)

Because the `null` provisioner performs no actual management tasks, no further orchestration steps are necessary as long as the node is marked as `is_provisioned: true`. We can see below that when we `provision` with the `null` provisioner, no action is actually performed.

> **Note:** Users are responsible for ensuring that manually provisioned nodes meet all system requirements and are fully compliant with the orchestrator library.

In [ ]:
from ifran import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

await prov.provision(quiet=False)

## SLICES BI Provisioner

The SLICES BI provisioner is used to create ephemeral virtual machines on the [SLICES Basic Infrastructure Service (SLICES BI)](https://doc.slices-ri.eu/BasicServices/BasicInfrastructure/basicinfrastructure.html).

Using this provisioner requires the SLICES CLI to be installed on your local machine. Refer to the [Preparation tutorial](./preparation.ipynb) for installation instructions and a review of fundamental SLICES-RI concepts.

> **Important:** Ensure your SSH keys are registered with SLICES before deploying your network.

### Configuring the Deployment Model

Having already defined the metamodel, you must now configure the deployment model to specify the target infrastructure for provisioning.

The `ProvisioningManagement` manager provisions the machines declared in the `nodes` section of the deployment model, automatically determining the correct provisioner for each node using the configurations defined in the `provisioners` section.

In this example, you will deploy three virtual machines (`node1`, `node2`, and `node3`) on the `be-gent1-bi-vm1` SLICES-BI infrastructure (you can list the available LICES-BI infrastructures with the `slices bi infra list --all`). These nodes will run within the `provision_tutorial` experiment under the `post5g-beta` project. The configuration instructs the provisioner to automatically create the experiment and assigns the nodes a 1-hour lifetime, after which they will be automatically destroyed. These parameters are defined in a custom provisioner named `my-provisioner` within the `provisioners` section of the deployment model.

Once the provisioner is defined, you declare the target nodes and link them to it:
* **`node1`**: Tagged with the `core` role.
* **`node2`**: Tagged with the `ran` role.
* **`node3`**: Tagged to support both `ran` and `ue` roles.

All three nodes are provisioned using `my-provisioner`.

In [23]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
provisioners:
  my-provisionner:
    type: bi
    params:
      infra: be-gent1-bi-vm1
      project: post5g-beta
      experiment: provision_tutorial
      create_experiment: true
      experiment_duration: 1h
      node_duration: 1h
nodes:
  node1:
    provisioner: my-provisionner
    roles:
    - core
  node2:
    provisioner: my-provisionner
    roles:
    - ran
  node3:
    provisioner: my-provisionner
    roles:
    - ran
    - ue
"""
)


To provision the nodes, instantiate a `ProvisioningManagement` manager using the previously defined metamodel and deployment model, then invoke its `provision()` method.

In [24]:
from ifran import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

await prov.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node3' provisioning] ***

TASK [provisioning/bi/provision : Select project 'post5g-beta' for '

Your three virtual machines are now successfully deployed on SLICES BI and configured for the scenario. You can verify the deployment by listing the experiment resources using the SLICES CLI (be sure to adjust the parameters to match your specific setup):

In [26]:
!SLICES_BI_SITE_ID=be-gent1-bi-vm1 slices bi list-resources --experiment provision_tutorial

  Resources in experiment exp_expauth.ilabt.imec.be_01m78h9j6wekebpx0yh5e5qp10  
┏━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃           ┃       ┃        ┃ Created   ┃            ┃ Public    ┃ Private    ┃
┃ ID        ┃ Name  ┃ Status ┃ At        ┃ Expires At ┃ IPv4      ┃ IPv4       ┃
┡━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ r_be-gen… │ node2 │ up     │ 2026-11-… │ 2026-11-12 │           │ 10.10.220… │
│           │       │        │ 19:46 CET │ 20:46 CET  │           │            │
│ r_be-gen… │ node1 │ up     │ 2026-11-… │ 2026-11-12 │           │ 10.10.219… │
│           │       │        │ 19:46 CET │ 20:46 CET  │           │            │
│ r_be-gen… │ node3 │ up     │ 2026-11-… │ 2026-11-12 │           │ 10.10.216… │
│           │       │        │ 19:46 CET │ 20:46 CET  │           │            │
└───────────┴───────┴────────┴───────────┴────────────┴───────────┴────────────┘


## Saving the Metamodel and Deployment Model

Each manager accepts a metamodel and a deployment model as inputs and outputs an updated deployment model reflecting the results of its execution. This updated model can then be passed downstream to the next manager in the pipeline.

You can retrieve the updated metamodel and deployment model from a manager through several convenience methods. For instance, the `output()` method directly returns both models. The updated deployment model contains all technical credentials and parameters required to access the newly created virtual machines.

To persist the current infrastructure state for subsequent orchestration steps, you can save both models to file:

In [25]:
metamodel, deployment_model = prov.output()
from pathlib import Path
DIR="state/provision"
Path(DIR).mkdir(parents=True, exist_ok=True)

metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml", exclude_defaults=False)
deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml", exclude_defaults=False)

## Unprovisioning Nodes

Provisioners can also be used to unprovision the nodes they have previously deployed. 

You can choose to target specific nodes for unprovisioning or clean up all nodes declared within the deployment model at once.

The exact effect of unprovisioning depends on the underlying provisioner (for instance, the SLICES BI provisioner terminates and destroys the corresponding virtual machine). However, regardless of the provider, the node entry remains present in the deployment model—allowing it to be re-provisioned later—while its `is_provisioned` attribute is reset to `false`.

In [7]:
await prov.unprovision_nodes(nodes=["node1", "node2"])  # Unprovision some nodes
!SLICES_BI_SITE_ID=be-gent1-bi-vm1 slices bi list-resources --experiment provision_tutorial


await prov.unprovision()                                # Unprovision all nodes
!SLICES_BI_SITE_ID=be-gent1-bi-vm1 slices bi list-resources --experiment provision_tutorial

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/unprovision : Select project 'post5g-beta' for 'node1' unprovisioning] ***
TASK [provisioning/bi/unprovision : Select project 'post5g-beta' for 'node2' unprovisioning] ***

changed: [localhost]

TASK [provisioning/bi/unprovision : Destroy 'node1' on infrastructure 'be-gent1-bi-vm1' using experiment 'provision_tutorial'] ***
changed: [localhost]

TASK [provisioning/bi/unprovision : Destroy 'node2' on infrastructure '

## POS Provisioner

The POS provisioner is used to provision machines managed by the POS infrastructure.

There are two types of POS infrastructure: `legacy POS`, which is centralized, and newer `POS` infrastructures, which are decentralized. In the legacy setup, actions are performed directly on the central POS machine. In newer setups, actions are performed without relying on a central machine.

Resources in POS infrastructures must be reserved before they can be provisioned. The provisioner can attempt to reserve a resource on behalf of the user by setting `attempt_reservation` to `true`. Otherwise, users must reserve the resource before calling the provisioner.

POS allows you to specify both the Linux image to run on the machine and the boot parameters (i.e., `cmdline`). The example below demonstrates a combination of these options.

To use this provisioner, you must have the SLICES CLI installed on your local machine, including the POS subcommand. Refer to the Preparation tutorial for installation instructions and an overview of the fundamental SLICES-RI concepts.

> **Important:** Ensure that your SSH keys are registered with SLICES before deploying your network.

> **Important:** If you are using legacy POS, ensure that your local machine is configured with an SSH key registered with Duckburg.

In [ ]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
provisioners:
  pos:
    type: pos
    params:
      testbed: sophia
      attempt_reservation: true

  legacy_pos:
    type: legacy_pos
    params:
      attempt_reservation: false
      central_pos_ssh_info:
        hostname: duckburg.net.cit.tum.de
        port: 10022
        user: <USERNAME>

nodes:
  sopnode-f1:
    provisioner: pos
    provisioning_params:
      image: ubuntu-noble
      boot_params: >-
        biosdevname=0 net.ifnames=0 mitigations=off intel_iommu=on iommu=pt
        selinux=0 enforcing=0 isolcpus=managed_irq,16-63
        nohz_full=16-63 nohz=on rcu_nocbs=16-63
        kthread_cpus=0-4 irqaffinity=0-4 rcu_nocb_poll
        nosoftlockup intel_pstate=disable idle=poll skew_tick=1
        tsc=nowatchdog nmi_watchdog=0 softlockup_panic=0 audit=0 nosmt   

  sopnode-f2:
    provisioner: pos
    provisioning_params:
      image: ubuntu-jammy

  standard-2-1:
    provisioner: legacy_pos   
"""
)

We can then provision the nodes. This may take a while as the procedures requires a full hardware reboot of the nodes.

In [10]:
from ifran import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

await prov.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

TASK [provisioning/pos/reserve : Select testbed 'sophia'] **********************
changed: [localhost]

TASK [provisioning/pos/reserve : Reserve 'sopnode-f1, sopnode-f2' for '120' minutes] ***
changed: [localhost]

PLAY RECAP *********************************************************************
localhost                  : ok=2    changed=2    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, o

We can now check that the machines have been started properly.

If we use `legacy pos` we first have to connect to the central node (in our case `duckburg.net.in.tum.de`) and execute command from there directly. 

As we can see the machine has just started.

In [ ]:
! ssh -o StrictHostKeyChecking=no <USERNAME>@duckburg.net.in.tum.de -p 10022 "ssh standard-2-1 uptime"

 19:29:08 up 3 min,  1 user,  load average: 0.16, 0.03, 0.01


When using the `pos` provisioner, we can use the `slices pos ssh --show <NODE>` command to obtain the SSH command to connect to the node.

Again we see that the machine has just started

In [13]:
! slices pos ssh sopnode-f1 uptime

 19:31:54 up 4 min,  1 user,  load average: 0.02, 0.08, 0.04


**NOTE:** to simplify execution one can use them as jump hosts.

We can also check that the image and bootparameters have been set properly.

In [14]:
! slices pos ssh sopnode-f1 "cat /etc/os-release"

PRETTY_NAME="Ubuntu 24.04 LTS"
NAME="Ubuntu"
VERSION_ID="24.04"
VERSION="24.04 LTS (Noble Numbat)"
VERSION_CODENAME=noble
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=noble
LOGO=ubuntu-logo


In [15]:
! slices pos ssh sopnode-f1 "cat /proc/cmdline"

vmlinuz boot=live vga=791 lvm dhcp noeject noprompt biosdevname=0 net.ifnames=0 mitigations=off intel_iommu=on iommu=pt selinux=0 enforcing=0 isolcpus=managed_irq,16-63 nohz_full=16-63 nohz=on rcu_nocbs=16-63 kthread_cpus=0-4 irqaffinity=0-4 rcu_nocb_poll nosoftlockup intel_pstate=disable idle=poll skew_tick=1 tsc=nowatchdog nmi_watchdog=0 softlockup_panic=0 audit=0 nosmt NODE_ID=sopnode-f1 POS=http://172.28.2.120:5000 BOOTIF=01-30-3e-a7-18-00-00 fetch=http://172.28.2.120:5000/pxe/files/default/ubuntu-noble/image.squashfs hostname=sopnode-f1


In [16]:
! slices pos ssh sopnode-f2 "cat /etc/os-release"

PRETTY_NAME="Ubuntu 22.04 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04 (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


In [17]:
! slices pos ssh sopnode-f2 "cat /proc/cmdline"

vmlinuz boot=live vga=791 lvm dhcp noeject noprompt NODE_ID=sopnode-f2 POS=http://172.28.2.120:5000 BOOTIF=01-30-3e-a7-18-24-00 fetch=http://172.28.2.120:5000/pxe/files/default/ubuntu-jammy/image.squashfs hostname=sopnode-f2


For convenience, we can run commands on the provisioned machines directly from the library using the `run_node_command` function:

In [20]:
print(await prov.run_node_command(command=["cat", "/etc/os-release"], nodes=["sopnode-f2"], quiet=True))

PRETTY_NAME="Ubuntu 22.04 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04 (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


Additionaly, to copy files to a provisioned machine, we can use the `node_copy` functions:

In [21]:
await prov.node_copy(local_path="./state/provision/metamodel.prov.yaml", remote_path="/root", nodes=["standard-2-1"], quiet=True)

We can check it the file was copied successfully:

In [22]:
print(await prov.run_node_command(command=["cat", "/root/metamodel.prov.yaml"], nodes=["standard-2-1"], quiet=True))

core:
  components:
    amf:
      category: core
      type: amf
      config:
        sbi:
          interfaceName: eth0
          ipv4EndPointAddress: 0.0.0.0
          ipv6EndPointAddress: null
          port: 80
          protocol: TCP
          scheme: http
          apiVersion: v1
          tls:
            enabled: false
            certificate: null
            privateKey: null
        nrfUri: null
        n2:
          interfaceName: eth0
          ipv4EndPointAddress: 0.0.0.0
          ipv6EndPointAddress: null
          port: 38412
          protocol: SCTP
        guamiList:
        - plmnId:
            mcc: '001'
            mnc: '01'
            mncLength: 2
            plmnId: '00101'
          amfRegionId: '02'
          amfSetId: '001'
          amfPointer: '00'
          amfId: '020040'
        taiList:
        - plmnId:
            mcc: '001'
            mnc: '01'
            mncLength: 2
            plmnId: '00101'
          tac: 1
        plmnList:
        - plmnI